# Phase 1: XAI Infrastructure Verification

This notebook verifies that the XAI infrastructure (`xai/` package) correctly loads the model,
reproduces training-time predictions, and can extract all intermediate tensors needed by
Phases 2–8 (Grad-CAM, Attention, SHAP, LIME, Case Studies, Reports, Thesis Figures).

| Component | Configuration |
|---|---|
| Image Backbone | Swin-B (`swin_base_patch4_window7_224`) |
| Text Backbone | PhoBERT (`vinai/phobert-base-v2`) |
| Fusion | Cross-Attention (8 heads, hidden=512) |
| Loss | LogCosh |

> **10 verification checks (V1–V10) must all pass before proceeding to Phase 2.**

---
### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone source code and install dependencies

The `xai/` package lives on the `visualization` branch.

In [ ]:
!git clone -b visualization https://github.com/lechihoang/SE365.git /content/SE365 2>/dev/null || echo 'Repo already cloned'
%cd /content/SE365
!pip install -q -r requirements.txt

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configuration

**Edit this cell only.** All paths are derived from these variables.

In [ ]:
import os
import sys
import time
import math
import warnings
warnings.filterwarnings('ignore')

# ── User configuration (edit here) ──
DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

# ── Derived paths (do not edit) ──
EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_OUT_DIR  = f'{DRIVE_ROOT}/xai/phase1'
DATA_DIR     = f'{PROJECT_ROOT}/data/text'
IMAGE_DIR    = f'{PROJECT_ROOT}/data/image'

os.makedirs(XAI_OUT_DIR, exist_ok=True)

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Verify xai/ package exists after clone
xai_pkg = os.path.join(PROJECT_ROOT, 'xai', '__init__.py')
if not os.path.isfile(xai_pkg):
    raise FileNotFoundError(
        f'xai/ package not found at {PROJECT_ROOT}/xai/.\n'
        f'Make sure STEP 2 cloned the visualization branch:\n'
        f'  !git clone -b visualization https://github.com/lechihoang/SE365.git /content/SE365'
    )

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DRIVE_ROOT   : {DRIVE_ROOT}')
print(f'EXP_ID       : {EXP_ID}')
print(f'EXP_DIR      : {EXP_DIR}')
print(f'XAI_OUT_DIR  : {XAI_OUT_DIR}')
print(f'DATA_DIR     : {DATA_DIR}')
print(f'IMAGE_DIR    : {IMAGE_DIR}')
print(f'CWD          : {os.getcwd()}')
print(f'xai/ package : OK')

### STEP 5: Imports and Seed (V1)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 5/14 — Imports and Seed (V1)')
print('='*60)

import json
import numpy as np
import pandas as pd
import torch

from xai.config import (
    TARGET_NAMES, TARGET_INDICES, FACTOR_NAMES, DISPLAY_NAMES, LABEL_COLS,
    DEFAULT_SEED, DEFAULT_DPI, THESIS_DPI,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL, BEST_FUSION_TYPE, BEST_EXP_ID,
    TEXT_FEATURE_DIM, IMAGE_FEATURE_DIM, FUSED_DIM, NUM_TARGETS,
    PHOBERT_NUM_LAYERS, PHOBERT_NUM_HEADS,
    COLOR_SCHEMES, INDEX_TO_FACTOR, FACTOR_TO_INDEX,
)
from xai.utils import (
    get_device, set_seed, get_tokenizer, get_image_processor,
    load_model, load_single_sample, get_prediction,
    save_figure, save_raw_values, get_metadata,
)

SEED = DEFAULT_SEED
set_seed(SEED)
device = get_device()

# Track verification results
verification = {}

assert str(device) in ('cuda', 'mps', 'cpu'), f'Unexpected device: {device}'
verification['v1_seed_device'] = True

print(f'Device       : {device}')
print(f'Seed         : {SEED}')
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA avail   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
print(f'\nV1: Seed and device — PASSED  ({time.time()-t0:.1f}s)')

### STEP 6: Load Model (V2)

After loading, we patch the text encoder to use **eager** attention (instead of sdpa)
so that `output_attentions=True` returns real attention weights in V8.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 6/14 — Load Model (V2)')
print('='*60)

model, config = load_model(EXP_DIR, device=device)

# ── Patch sdpa -> eager attention for XAI ──
# Recent transformers uses sdpa by default, which does not return attention weights.
# We replace each SdpaSelfAttention with the eager RobertaSelfAttention in-place.
# Weights are identical; only the computation path changes.
encoder = model.text_model.encoder
if hasattr(encoder, 'config'):
    encoder.config._attn_implementation = 'eager'
    encoder.config.attn_implementation = 'eager'

patched = 0
try:
    from transformers.models.roberta.modeling_roberta import RobertaSelfAttention
    if hasattr(encoder, 'encoder') and hasattr(encoder.encoder, 'layer'):
        for layer_module in encoder.encoder.layer:
            attn = layer_module.attention.self
            cls_name = type(attn).__name__
            if 'Sdpa' in cls_name or 'Flash' in cls_name:
                eager_attn = RobertaSelfAttention(encoder.config)
                eager_attn.load_state_dict(attn.state_dict())
                eager_attn.to(next(attn.parameters()).device)
                layer_module.attention.self = eager_attn
                patched += 1
except Exception as e:
    print(f'[XAI] WARNING: Could not patch attention: {e}')

if patched > 0:
    print(f'[XAI] Patched {patched} attention layers: sdpa -> eager')
else:
    print('[XAI] Attention already eager (or no patch needed)')

from Models.CrossAttentionFusion import CrossAttentionFusion
assert isinstance(model, CrossAttentionFusion), f'Expected CrossAttentionFusion, got {type(model)}'
assert model.text_model.encoder.config.hidden_size == TEXT_FEATURE_DIM
assert model.image_model.encoder.num_features == IMAGE_FEATURE_DIM
assert not model.training, 'Model must be in eval mode'

total_params = sum(p.numel() for p in model.parameters())
verification['v2_model_loading'] = True

print(f'Model class  : {model.__class__.__name__}')
print(f'Parameters   : {total_params:,}')
print(f'Eval mode    : {not model.training}')
print(f'text_dim     : {model.text_model.encoder.config.hidden_size}')
print(f'image_dim    : {model.image_model.encoder.num_features}')
if config.get('_best_mean_mae'):
    print(f'Best MAE     : {config["_best_mean_mae"]:.4f}')
print(f'\nV2: Model loading — PASSED  ({time.time()-t0:.1f}s)')

### STEP 7: Tokenizer and Image Processor (V3)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 7/14 — Tokenizer & Image Processor (V3)')
print('='*60)

text_model_name = config.get('text_model_name', BEST_TEXT_MODEL)
image_model_name = config.get('image_model_name', BEST_IMAGE_MODEL)

tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

# verify tokenizer
test_enc = tokenizer('Xin chao', return_tensors='pt')
assert test_enc['input_ids'].numel() > 0, 'Tokenizer produced empty output'

# verify image processor
from PIL import Image as PILImage
test_img = PILImage.new('RGB', (224, 224), color='black')
test_pix = image_processor([test_img], return_tensors='pt')['pixel_values']
assert test_pix.shape[-2:] == (224, 224), f'Unexpected image shape: {test_pix.shape}'

verification['v3_tokenizer_processor'] = True

print(f'Tokenizer    : {text_model_name} (vocab={tokenizer.vocab_size})')
print(f'Img processor: {image_model_name}')
print(f'Test token   : {test_enc["input_ids"].shape}')
print(f'Test image   : {test_pix.shape}')
print(f'\nV3: Tokenizer & image processor — PASSED  ({time.time()-t0:.1f}s)')

### STEP 8: Load Single Sample (V4)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 8/14 — Load Single Sample (V4)')
print('='*60)

# Use test set if test_predictions.csv exists, else validation set
test_csv = os.path.join(DATA_DIR, 'test.csv')
val_csv  = os.path.join(DATA_DIR, 'val.csv')
test_pred_path = os.path.join(EXP_DIR, 'test_predictions.csv')
val_pred_path  = os.path.join(EXP_DIR, 'predictions.csv')

if os.path.isfile(test_pred_path) and os.path.isfile(test_csv):
    SPLIT_CSV = test_csv
    PRED_CSV  = test_pred_path
    SPLIT_NAME = 'test'
else:
    SPLIT_CSV = val_csv
    PRED_CSV  = val_pred_path
    SPLIT_NAME = 'validation'

print(f'Using split  : {SPLIT_NAME}')
print(f'Split CSV    : {SPLIT_CSV}')
print(f'Predictions  : {PRED_CSV}')

sample = load_single_sample(
    csv_path=SPLIT_CSV, idx=0,
    tokenizer=tokenizer, image_processor=image_processor,
    image_dir=IMAGE_DIR, device=device,
)

max_len = config.get('max_length', 256)
assert sample['input_ids'].shape      == (1, max_len), f'input_ids: {sample["input_ids"].shape}'
assert sample['attention_mask'].shape  == (1, max_len), f'attention_mask: {sample["attention_mask"].shape}'
assert sample['pixel_values'].shape[0] == 1 and sample['pixel_values'].shape[1] == 4
assert sample['num_images'].shape      == (1,)
assert sample['factor_scores'].shape   == (5,)
assert len(sample['text']) > 0

verification['v4_single_sample'] = True

print(f'input_ids    : {sample["input_ids"].shape}')
print(f'attention_mask: {sample["attention_mask"].shape}')
print(f'pixel_values : {sample["pixel_values"].shape}')
print(f'num_images   : {sample["num_images"].item()} real images')
print(f'Text (100ch) : {sample["text"][:100]}...')
print(f'Ground truth : {[f"{v:.2f}" for v in sample["factor_scores"].tolist()]}')
print(f'\nV4: Single sample loading — PASSED  ({time.time()-t0:.1f}s)')

### STEP 9: Run Inference (V5)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 9/14 — Run Inference (V5)')
print('='*60)

result = get_prediction(model, sample)

assert len(result['predictions']) == NUM_TARGETS
assert np.isfinite(result['predictions_array']).all(), 'Predictions contain NaN/Inf'

verification['v5_inference'] = True

print(f'{"Target":<25s} {"Pred":>8s} {"GT":>8s} {"Error":>8s}')
print('-' * 52)
for name in TARGET_NAMES:
    p = result['predictions'][name]
    g = result['ground_truth'][name]
    e = result['absolute_errors'][name]
    print(f'{name:<25s} {p:8.4f} {g:8.2f} {e:8.4f}')
print(f'{"":<25s} {"":>8s} {"":>8s} {"------":>8s}')
print(f'{"Mean MAE":<25s} {"":>8s} {"":>8s} {result["mean_mae"]:8.4f}')
print(f'\nV5: Single-sample inference — PASSED  ({time.time()-t0:.1f}s)')

### STEP 10: Numerical Match Against predictions.csv (V6)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 10/14 — Numerical Consistency (V6)')
print('='*60)

if not os.path.isfile(PRED_CSV):
    print(f'WARNING: {PRED_CSV} not found. Skipping V6.')
    verification['v6_numerical_consistency'] = {'passed': False, 'reason': 'predictions CSV missing'}
else:
    df_pred = pd.read_csv(PRED_CSV)

    use_amp = config.get('use_amp', False)
    TOLERANCE = 1e-3 if use_amp else 1e-4

    verify_indices = [0, 1, 2]
    max_diff_all = 0.0
    all_passed = True

    print(f'Tolerance    : {TOLERANCE} (AMP={use_amp})')
    print(f'Verifying    : {len(verify_indices)} samples\n')

    for sidx in verify_indices:
        if sidx >= len(df_pred):
            continue
        s = load_single_sample(
            csv_path=SPLIT_CSV, idx=sidx,
            tokenizer=tokenizer, image_processor=image_processor,
            image_dir=IMAGE_DIR, device=device,
        )
        r = get_prediction(model, s)
        row = df_pred.iloc[sidx]
        diffs = []
        for fi, fname in enumerate(FACTOR_NAMES):
            stored = float(row[f'y_pred_{fname}'])
            computed = r['predictions_array'][fi]
            diff = abs(stored - computed)
            diffs.append(diff)
            max_diff_all = max(max_diff_all, diff)
        sample_max = max(diffs)
        status = 'OK' if sample_max < TOLERANCE else 'FAIL'
        if sample_max >= TOLERANCE:
            all_passed = False
        print(f'  Sample {sidx}: max_diff={sample_max:.6f} [{status}]')

    verification['v6_numerical_consistency'] = {
        'passed': all_passed,
        'max_diff': float(max_diff_all),
        'tolerance': TOLERANCE,
        'num_samples_verified': len(verify_indices),
        'use_amp': use_amp,
    }

    if all_passed:
        print(f'\nV6: Numerical consistency — PASSED  (max_diff={max_diff_all:.6f})  ({time.time()-t0:.1f}s)')
    else:
        print(f'\nV6: Numerical consistency — FAILED  (max_diff={max_diff_all:.6f} > tol={TOLERANCE})')
        print('    May be caused by device differences (GPU training vs CPU inference).')
        print('    If max_diff < 1e-2, explanations are still valid.')

### STEP 11: Intermediate Tensors (V7)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 11/14 — Intermediate Tensors (V7)')
print('='*60)

with torch.no_grad():
    _, text_features = model.text_model(
        sample['input_ids'], sample['attention_mask']
    )
    _, image_features = model.image_model(
        sample['pixel_values'], num_images=sample['num_images']
    )

assert text_features.shape  == (1, TEXT_FEATURE_DIM), f'text_features: {text_features.shape}'
assert image_features.shape == (1, IMAGE_FEATURE_DIM), f'image_features: {image_features.shape}'
assert torch.isfinite(text_features).all(),  'text_features contain NaN/Inf'
assert torch.isfinite(image_features).all(), 'image_features contain NaN/Inf'

verification['v7_intermediate_tensors'] = {
    'text_features_shape': list(text_features.shape),
    'image_features_shape': list(image_features.shape),
}

print(f'text_features  : {text_features.shape}  mean={text_features.mean():.4f}  std={text_features.std():.4f}')
print(f'image_features : {image_features.shape}  mean={image_features.mean():.4f}  std={image_features.std():.4f}')
print(f'\nV7: Intermediate tensor extraction — PASSED  ({time.time()-t0:.1f}s)')

### STEP 12: PhoBERT Attention Extraction (V8)

Step 6 already patched sdpa → eager attention. This cell verifies that
`output_attentions=True` now returns real attention weight matrices.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 12/14 — PhoBERT Attention (V8)')
print('='*60)

v8_passed = False
v8_info = {}

try:
    with torch.no_grad():
        text_outputs = model.text_model.encoder(
            input_ids=sample['input_ids'],
            attention_mask=sample['attention_mask'],
            output_attentions=True,
            return_dict=True,
        )

    attentions = text_outputs.attentions

    if attentions is None or len(attentions) == 0:
        print('WARNING: output_attentions=True returned empty attentions.')
        print('  The text encoder may still be using sdpa attention.')
        print('  Check that the eager attention patch in Step 6 succeeded.')
        v8_info = {'passed': False, 'reason': 'attentions empty (sdpa still active?)'}
    else:
        num_layers = len(attentions)
        num_heads = attentions[0].shape[1]
        seq_len = sample['input_ids'].shape[1]
        expected_shape = (1, num_heads, seq_len, seq_len)

        assert attentions[0].shape == expected_shape, (
            f'Expected {expected_shape}, got {attentions[0].shape}'
        )
        assert (attentions[-1] >= 0).all(), 'Negative attention values found'

        row_sums = attentions[-1][0, 0].sum(dim=-1)
        assert torch.allclose(
            row_sums, torch.ones_like(row_sums), atol=1e-4
        ), 'Attention rows do not sum to 1'

        v8_passed = True
        v8_info = {
            'passed': True,
            'num_layers': num_layers,
            'num_heads': num_heads,
            'attention_shape_per_layer': list(attentions[0].shape),
        }

        print(f'Layers       : {num_layers}')
        print(f'Heads        : {num_heads}')
        print(f'Shape/layer  : {attentions[0].shape}')
        print(f'Values >= 0  : True')
        print(f'Row sums ~1  : True')

except Exception as e:
    print(f'ERROR during attention extraction: {e}')
    v8_info = {'passed': False, 'reason': str(e)}

verification['v8_attention_extraction'] = v8_info

if v8_passed:
    print(f'\nV8: PhoBERT attention extraction — PASSED  ({time.time()-t0:.1f}s)')
else:
    print(f'\nV8: PhoBERT attention extraction — FAILED')
    print(f'    Reason: {v8_info.get("reason", "unknown")}')

### STEP 13: Swin-B Spatial Feature Map via Hook (V9)

Probes the image encoder for spatial feature maps and normalizes them to
`[B, C, H, W]` format for Grad-CAM. Handles all timm output formats:
BCHW, BHWC (Swin-B default), BNC, BCN.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 13/14 — Swin-B Spatial Feature Map (V9)')
print('='*60)

# ── Helper: normalize any feature map format to [B, C, H, W] ──
def normalize_feature_map_to_bchw(tensor, expected_channels):
    """Convert spatial feature map from any timm format to [B, C, H, W]."""
    shape = tensor.shape
    ndim = len(shape)
    meta = {
        'raw_shape': list(shape),
        'detected_format': None,
        'reshape_required': False,
        'reshape_rule': None,
        'target_shape': None,
    }

    if ndim == 4:
        B, d1, d2, d3 = shape
        # [B, C, H, W]
        if d1 == expected_channels and d2 > 1 and d3 > 1:
            meta['detected_format'] = 'BCHW'
            meta['target_shape'] = list(shape)
            return tensor, meta
        # [B, H, W, C]  — Swin-B default in timm
        if d3 == expected_channels and d1 > 1 and d2 > 1:
            out = tensor.permute(0, 3, 1, 2).contiguous()
            meta['detected_format'] = 'BHWC'
            meta['reshape_required'] = True
            meta['reshape_rule'] = 'permute(0, 3, 1, 2)'
            meta['target_shape'] = list(out.shape)
            return out, meta

    elif ndim == 3:
        B, d1, d2 = shape
        # [B, N, C]
        if d2 == expected_channels:
            N = d1
            H = W = int(math.sqrt(N))
            if H * W == N:
                out = tensor.permute(0, 2, 1).reshape(B, expected_channels, H, W).contiguous()
                meta['detected_format'] = 'BNC'
                meta['reshape_required'] = True
                meta['reshape_rule'] = f'permute(0,2,1).reshape(B,{expected_channels},{H},{W})'
                meta['target_shape'] = list(out.shape)
                return out, meta
        # [B, C, N]
        if d1 == expected_channels:
            N = d2
            H = W = int(math.sqrt(N))
            if H * W == N:
                out = tensor.reshape(B, expected_channels, H, W).contiguous()
                meta['detected_format'] = 'BCN'
                meta['reshape_required'] = True
                meta['reshape_rule'] = f'reshape(B,{expected_channels},{H},{W})'
                meta['target_shape'] = list(out.shape)
                return out, meta

    raise ValueError(f'Cannot normalize shape {list(shape)} with expected_channels={expected_channels}')


# ── Probe candidate layers ──
hook_outputs = {}
hooks = []

def make_hook(name):
    def hook_fn(module, input, output):
        if isinstance(output, torch.Tensor):
            hook_outputs[name] = output.detach()
        elif isinstance(output, tuple) and len(output) > 0 and isinstance(output[0], torch.Tensor):
            hook_outputs[name] = output[0].detach()
    return hook_fn

img_encoder = model.image_model.encoder
candidates = {}
if hasattr(img_encoder, 'norm'):
    candidates['encoder.norm'] = img_encoder.norm
if hasattr(img_encoder, 'layers'):
    candidates['encoder.layers[-1]'] = img_encoder.layers[-1]
elif hasattr(img_encoder, 'stages'):
    candidates['encoder.stages[-1]'] = img_encoder.stages[-1]

for name, module in candidates.items():
    hooks.append(module.register_forward_hook(make_hook(name)))

single_image = sample['pixel_values'][0, 0:1]  # [1, C, H, W]
with torch.no_grad():
    _ = img_encoder(single_image)

for h in hooks:
    h.remove()

# ── Find and normalize best spatial feature map ──
expected_channels = model.image_model.encoder.num_features
best_layer = None
best_meta = None

print(f'Expected channels: {expected_channels}')
print('Candidate layers:')

for name, tensor in hook_outputs.items():
    if tensor is None:
        print(f'  {name}: None (skipped)')
        continue
    print(f'  {name}: {list(tensor.shape)}')
    try:
        bchw, meta = normalize_feature_map_to_bchw(tensor, expected_channels)
        print(f'    -> {meta["detected_format"]} -> {list(bchw.shape)}')
        if best_layer is None or name == 'encoder.norm':
            best_layer = name
            best_meta = meta
    except ValueError as e:
        print(f'    -> Cannot normalize: {e}')

assert best_layer is not None, (
    f'Could not find spatial feature map! '
    f'Captured: {[(n, list(t.shape)) for n, t in hook_outputs.items() if t is not None]}'
)

verification['v9_spatial_feature_map'] = {
    'passed': True,
    'target_layer_name': best_layer,
    'raw_shape': best_meta['raw_shape'],
    'detected_format': best_meta['detected_format'],
    'reshape_required': best_meta['reshape_required'],
    'reshape_rule': best_meta['reshape_rule'],
    'target_shape': best_meta['target_shape'],
}

print(f'\nSelected layer   : {best_layer}')
print(f'Raw shape        : {best_meta["raw_shape"]}')
print(f'Detected format  : {best_meta["detected_format"]}')
print(f'Grad-CAM shape   : {best_meta["target_shape"]}  [B, C, H, W]')
if best_meta['reshape_rule']:
    print(f'Reshape rule     : {best_meta["reshape_rule"]}')
print(f'\nV9: Swin-B spatial feature map — PASSED  ({time.time()-t0:.1f}s)')

### STEP 14: Forward-Pass Consistency (V10)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 1 — Step 14/14 — Forward-Pass Consistency (V10)')
print('='*60)

with torch.no_grad():
    preds_full = model(
        input_ids=sample['input_ids'],
        attention_mask=sample['attention_mask'],
        pixel_values=sample['pixel_values'],
        num_images=sample['num_images'],
    )
    if isinstance(preds_full, tuple):
        preds_full = preds_full[0]

with torch.no_grad():
    _, tf = model.text_model(sample['input_ids'], sample['attention_mask'])
    _, imf = model.image_model(sample['pixel_values'], num_images=sample['num_images'])
    tf = tf.float()
    imf = imf.float()
    t = model.text_proj(tf).unsqueeze(1)
    i = model.image_proj(imf).unsqueeze(1)
    t_out, _ = model.cross_attn_t2i(query=t, key=i, value=i)
    i_out, _ = model.cross_attn_i2t(query=i, key=t, value=t)
    fused = torch.cat([t_out.squeeze(1), i_out.squeeze(1)], dim=1)
    preds_manual = model.head(fused)

max_diff = (preds_full - preds_manual).abs().max().item()
assert max_diff < 1e-5, f'Forward-pass mismatch: max_diff={max_diff}'

verification['v10_forward_consistency'] = {
    'passed': True,
    'max_diff': float(max_diff),
    'fused_shape': list(fused.shape),
}

print(f'Full forward   : {preds_full.cpu().numpy().flatten()}')
print(f'Manual forward : {preds_manual.cpu().numpy().flatten()}')
print(f'Max difference : {max_diff:.2e}')
print(f'Fused shape    : {fused.shape}  (first 512=text-origin, last 512=image-origin)')
print(f'\nV10: Forward-pass consistency — PASSED  ({time.time()-t0:.1f}s)')

---
### Save Verification Report

In [ ]:
print('='*60)
print('  Saving Verification Report')
print('='*60)

meta = get_metadata(exp_id=EXP_ID, config=config, device=device, seed=SEED)

report = {
    'phase': 'Phase 1: Infrastructure',
    **meta,
    'total_parameters': total_params,
    'model_class': model.__class__.__name__,
    'split_used': SPLIT_NAME,
    'verifications': verification,
}

report_path = os.path.join(XAI_OUT_DIR, 'verification_report.json')
save_raw_values(report, report_path)

sample_pred = {
    'phase': 'Phase 1: Infrastructure',
    **meta,
    'sample_idx': sample['sample_idx'],
    'split': SPLIT_NAME,
    'text': sample['text'],
    'num_images': sample['num_real_images'],
    'predictions': result['predictions'],
    'ground_truth': result['ground_truth'],
    'absolute_errors': result['absolute_errors'],
    'mean_mae': result['mean_mae'],
}
pred_path = os.path.join(XAI_OUT_DIR, 'sample_prediction.json')
save_raw_values(sample_pred, pred_path)

import importlib
env_info = {
    'phase': 'Phase 1: Infrastructure',
    **meta,
    'python_version': sys.version,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
}
for lib_name in ['transformers', 'timm', 'PIL', 'matplotlib']:
    try:
        lib = importlib.import_module(lib_name)
        env_info[f'{lib_name}_version'] = getattr(lib, '__version__', 'unknown')
    except ImportError:
        env_info[f'{lib_name}_version'] = 'not installed'
env_path = os.path.join(XAI_OUT_DIR, 'environment.json')
save_raw_values(env_info, env_path)

---
### Verification Summary

In [ ]:
print('='*60)
print('  PHASE 1 INFRASTRUCTURE VERIFICATION SUMMARY')
print('='*60)

checks = [
    ('V1',  'Seed and device',              'v1_seed_device'),
    ('V2',  'Model loading',                'v2_model_loading'),
    ('V3',  'Tokenizer & image processor',  'v3_tokenizer_processor'),
    ('V4',  'Single sample loading',        'v4_single_sample'),
    ('V5',  'Single-sample inference',       'v5_inference'),
    ('V6',  'Numerical consistency',         'v6_numerical_consistency'),
    ('V7',  'Intermediate tensors',          'v7_intermediate_tensors'),
    ('V8',  'PhoBERT attention',             'v8_attention_extraction'),
    ('V9',  'Swin-B feature map',           'v9_spatial_feature_map'),
    ('V10', 'Forward-pass consistency',      'v10_forward_consistency'),
]

all_passed = True
for vid, desc, key in checks:
    val = verification.get(key, False)
    if isinstance(val, dict):
        passed = val.get('passed', False)
    else:
        passed = bool(val)

    status = 'PASSED' if passed else 'FAILED'
    if not passed:
        all_passed = False

    extra = ''
    if key == 'v6_numerical_consistency' and isinstance(val, dict):
        extra = f'  (max_diff={val.get("max_diff", "?")})'
    elif key == 'v9_spatial_feature_map' and isinstance(val, dict) and val.get('passed'):
        extra = f'  (format={val.get("detected_format", "?")}, shape={val.get("target_shape", "?")})'
    elif key == 'v8_attention_extraction' and isinstance(val, dict) and val.get('passed'):
        extra = f'  ({val.get("num_layers")} layers, {val.get("num_heads")} heads)'

    print(f'  [{status:6s}] {vid:<4s} {desc}{extra}')

print('='*60)
if all_passed:
    print('  All verifications PASSED.')
    print('  Phase 1 infrastructure is ready for Phases 2-8.')
else:
    print('  Some verifications FAILED. Review the output above.')
print('='*60)

print(f'\nArtifacts saved to: {XAI_OUT_DIR}/')
for fname in ['verification_report.json', 'sample_prediction.json', 'environment.json']:
    fpath = os.path.join(XAI_OUT_DIR, fname)
    status = 'OK' if os.path.isfile(fpath) else 'MISSING'
    print(f'  [{status}] {fname}')